# 01 · Data preparation and task definition

This notebook establishes the data foundation for the HIV-1 drug-resistance prediction analysis. It loads the prepared Stanford HIVDB-derived drug-class files, converts them into a unified long-format task table, summarizes drug-level label distributions, checks sequence quality, and writes standardized artifacts for downstream notebooks.

The main output is `task_table.csv`, a shared table in which each row represents one `sequence × drug` prediction instance. Downstream notebooks use this table to compare mutation encodings, protein language model representations, fusion models, OOD generalization, and interpretability analyses under a consistent task definition.

The benchmark covers five HIV-1 drug classes spanning four viral proteins: protease (PR) for protease inhibitors (PI), reverse transcriptase (RT) for nucleoside and non-nucleoside RT inhibitors (NRTI/NNRTI), integrase (IN) for integrase strand-transfer inhibitors (INI), and capsid (CA) for the capsid inhibitor (CAI). Each drug-specific task uses binary resistance labels, with resistant samples encoded as `1`, susceptible samples encoded as `0`, and missing labels excluded from that drug's task. Two low-sample tasks (CAB and LEN) are retained but flagged as sparse so downstream analyses can interpret them with the appropriate caution.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyArrowPatch, PathPatch
from matplotlib.path import Path as MplPath

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "work" / "prepared").exists() else CWD.parent
assert (PROJECT_ROOT / "work" / "prepared").exists(), f"cannot locate work/prepared from {CWD}"

PREPARED_DIR = PROJECT_ROOT / "work" / "prepared"
RESULTS_DIR = PROJECT_ROOT / "results" / "notebooks" / "01_data"
FIG_DIR = PROJECT_ROOT / "figures" / "mutation_aware" / "01_data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("project root :", PROJECT_ROOT)
print("prepared dir :", PREPARED_DIR)
print("results dir  :", RESULTS_DIR)
print("figures dir  :", FIG_DIR)

project root : /mnt/d/我的项目/生信包复现重构升级/projects/1模型推理型/变异-耐药性预测/HIV-ESM-2
prepared dir : /mnt/d/我的项目/生信包复现重构升级/projects/1模型推理型/变异-耐药性预测/HIV-ESM-2/work/prepared
results dir  : /mnt/d/我的项目/生信包复现重构升级/projects/1模型推理型/变异-耐药性预测/HIV-ESM-2/results/notebooks/01_data
figures dir  : /mnt/d/我的项目/生信包复现重构升级/projects/1模型推理型/变异-耐药性预测/HIV-ESM-2/figures/mutation_aware/01_data


In [ ]:
# ---- Design-system palette (validated; see dataviz skill) --------------------
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK2 = "#52514e"
INK3 = "#8a8984"
GRID = "#e6e5e1"

CLASS_ORDER = ["PI", "NRTI", "NNRTI", "INI", "CAI"]
CLASS_COLOR = {"PI": "#2a78d6", "NRTI": "#1baf7a", "NNRTI": "#eda100",
               "INI": "#9b5de5", "CAI": "#e05264"}
BLUE_RAMP = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 11, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
    "text.color": INK, "xtick.color": INK2, "ytick.color": INK2, "axes.grid": False,
    "font.family": "DejaVu Sans",
})
print("palette ready")

## 1 · Load drug-class files and build a unified task table

The five input CSV files share the layout `SeqID, gene, sequence, seq_len, <DRUG>_label ...`. This module converts the wide drug-class tables into a long-format task table with one row per `sequence × drug` prediction instance. The resulting `task_table.csv` becomes the common input for downstream analysis.

In [ ]:
CLASS_FILES = {"PI": "PI.csv", "NRTI": "NRTI.csv", "NNRTI": "NNRTI.csv",
               "INI": "INI.csv", "CAI": "CAI.csv"}
CLASS_GENE = {"PI": "PR", "NRTI": "RT", "NNRTI": "RT", "INI": "IN", "CAI": "CA"}
# Low-sample tasks retained in the table but flagged so downstream analyses can
# treat them cautiously (CAB minority class = 12, LEN n = 140 with ~70% dupes).
SPARSE_DRUGS = {"CAB", "LEN"}

raw_frames = {}
for cls, fname in CLASS_FILES.items():
    df = pd.read_csv(PREPARED_DIR / fname)
    raw_frames[cls] = df
    label_cols = [c for c in df.columns if c.endswith("_label")]
    print(f"{cls:>5s}: {len(df):>5d} isolates | gene={df['gene'].iloc[0]} | "
          f"{len(label_cols)} drugs -> {[c.removesuffix('_label') for c in label_cols]}")

In [ ]:
def to_long(cls: str, df: pd.DataFrame) -> pd.DataFrame:
    label_cols = [c for c in df.columns if c.endswith("_label")]
    long = df.melt(
        id_vars=["SeqID", "gene", "sequence", "seq_len"],
        value_vars=label_cols,
        var_name="drug",
        value_name="label",
    )
    long["drug"] = long["drug"].str.removesuffix("_label")
    long["drug_class"] = cls
    long = long.dropna(subset=["label"]).copy()
    long["label"] = long["label"].astype(int)
    return long

task_table = pd.concat([to_long(c, raw_frames[c]) for c in CLASS_ORDER], ignore_index=True)
task_table["is_sparse"] = task_table["drug"].isin(SPARSE_DRUGS)
task_table = task_table[["SeqID", "drug_class", "drug", "gene", "seq_len", "label", "is_sparse", "sequence"]]
task_table["drug_class"] = pd.Categorical(task_table["drug_class"], CLASS_ORDER, ordered=True)

print("task instances :", len(task_table))
print("unique isolates:", task_table["SeqID"].nunique())
print("unique drugs   :", task_table["drug"].nunique())
print("sparse drugs   :", sorted(SPARSE_DRUGS))
task_table.head()

## 2 · Summarize sample size and resistance prevalence

This module calculates per-drug and per-class summary statistics, including valid sample size, resistant and susceptible counts, and resistance prevalence. These summaries document class imbalance and identify sparse or low-prevalence drug tasks that require careful interpretation in later modeling analyses.

In [ ]:
per_drug = (
    task_table.groupby(["drug_class", "drug"], observed=True)
    .agg(
        gene=("gene", "first"),
        seq_len=("seq_len", "median"),
        n_valid=("label", "size"),
        n_resistant=("label", "sum"),
        is_sparse=("is_sparse", "first"),
    )
    .reset_index()
)
per_drug["n_susceptible"] = per_drug["n_valid"] - per_drug["n_resistant"]
per_drug["prevalence"] = (per_drug["n_resistant"] / per_drug["n_valid"]).round(4)
per_drug = per_drug.sort_values(["drug_class", "prevalence"], ascending=[True, False]).reset_index(drop=True)
per_drug

In [ ]:
per_class = (
    per_drug.groupby("drug_class", observed=True)
    .agg(
        gene=("gene", "first"),
        n_drugs=("drug", "nunique"),
        n_sparse_drugs=("is_sparse", "sum"),
        n_task_instances=("n_valid", "sum"),
        median_prevalence=("prevalence", "median"),
        min_prevalence=("prevalence", "min"),
        max_prevalence=("prevalence", "max"),
    )
    .reset_index()
)
per_class

## 3 · Check sequence quality

This module summarizes sequence-level quality control for each drug class. It checks sequence-length consistency, non-standard amino-acid characters, and duplicate sequences. These checks determine whether downstream mutation encoding can align sequences to reference positions directly and identify cases that require handling during feature construction.

In [7]:
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

qc_rows = []
for cls in CLASS_ORDER:
    df = raw_frames[cls].drop_duplicates(subset=["SeqID"])
    seqs = df["sequence"].astype(str)
    lengths = seqs.str.len()
    nonstd = seqs.apply(lambda s: sum(ch not in STANDARD_AA for ch in s))
    qc_rows.append({
        "drug_class": cls, "gene": CLASS_GENE[cls], "n_isolates": len(df),
        "len_min": int(lengths.min()), "len_max": int(lengths.max()),
        "len_mode": int(lengths.mode().iloc[0]),
        "n_len_outliers": int((lengths != lengths.mode().iloc[0]).sum()),
        "n_seq_with_nonstd_aa": int((nonstd > 0).sum()),
        "pct_seq_with_nonstd_aa": round(100 * (nonstd > 0).mean(), 2),
        "n_duplicate_sequences": int(seqs.duplicated().sum()),
    })

seq_qc = pd.DataFrame(qc_rows)
seq_qc

,drug_class,gene,n_isolates,len_min,len_max,len_mode,n_len_outliers,n_seq_with_nonstd_aa,pct_seq_with_nonstd_aa,n_duplicate_sequences
0,PI,PR,2171,99,99,99,0,85,3.92,133
1,NRTI,RT,1844,239,240,240,4,153,8.30,32
2,NNRTI,RT,2272,317,318,318,1,232,10.21,146


## 4 · Save standardized artifacts

This module writes the standardized data artifacts used by later notebooks:

- `task_table.csv`: long-format task table with one `sequence × drug` prediction instance per row.
- `data_summary.csv`: per-drug sample size and resistance prevalence.
- `data_summary_by_class.csv`: per-class summary statistics.
- `sequence_qc_summary.csv`: sequence quality-control summary.
- `task_manifest.json`: compact metadata describing drugs, classes, and task counts.

In [ ]:
task_table_path = RESULTS_DIR / "task_table.csv"
task_table.drop(columns=["sequence"]).to_csv(task_table_path, index=False)
per_drug.to_csv(RESULTS_DIR / "data_summary.csv", index=False)
per_class.to_csv(RESULTS_DIR / "data_summary_by_class.csv", index=False)
seq_qc.to_csv(RESULTS_DIR / "sequence_qc_summary.csv", index=False)

manifest = {
    "n_task_instances": int(len(task_table)),
    "n_unique_isolates": int(task_table["SeqID"].nunique()),
    "n_drugs": int(task_table["drug"].nunique()),
    "sparse_drugs": sorted(SPARSE_DRUGS),
    "classes": {
        cls: {"gene": CLASS_GENE[cls],
              "drugs": per_drug.loc[per_drug["drug_class"] == cls, "drug"].tolist(),
              "sparse_drugs": per_drug.loc[(per_drug["drug_class"] == cls) & per_drug["is_sparse"], "drug"].tolist()}
        for cls in CLASS_ORDER
    },
}
(RESULTS_DIR / "task_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("wrote:")
for p in sorted(RESULTS_DIR.glob("*")):
    print("  ", p.relative_to(PROJECT_ROOT))

## 5 · Figure 1: drug-by-attribute matrix

This figure summarizes the data conditions for all 24 drug tasks in a compact matrix. Each row represents one drug, and the columns show sample size, resistance prevalence, and protein region. The matrix highlights task heterogeneity, including low-prevalence drugs such as DDI and TDF, the small sample size of RPV, and the sparse INI/CAI tasks CAB and LEN (marked with `*`).

In [ ]:
def blue_for(value: float, vmin: float, vmax: float) -> str:
    if vmax <= vmin:
        return BLUE_RAMP[2]
    frac = (value - vmin) / (vmax - vmin)
    idx = int(round(frac * (len(BLUE_RAMP) - 1)))
    return BLUE_RAMP[min(max(idx, 0), len(BLUE_RAMP) - 1)]


def ink_on(hexcolor: str) -> str:
    r, g, b = (int(hexcolor[i:i + 2], 16) / 255 for i in (1, 3, 5))
    lum = 0.2126 * r + 0.7152 * g + 0.0722 * b
    return "#ffffff" if lum < 0.55 else INK

mat = per_drug.copy()
n = len(mat)
fig, ax = plt.subplots(figsize=(8.6, 0.42 * n + 1.8))

n_min, n_max = mat["n_valid"].min(), mat["n_valid"].max()
p_min, p_max = mat["prevalence"].min(), mat["prevalence"].max()

for row, (_, r) in enumerate(mat.iloc[::-1].reset_index(drop=True).iterrows()):
    y = row
    c = blue_for(r["n_valid"], n_min, n_max)
    ax.add_patch(plt.Rectangle((0.0, y - 0.42), 0.92, 0.84, facecolor=c, edgecolor=SURFACE, lw=2))
    ax.text(0.46, y, f"{int(r['n_valid'])}", ha="center", va="center", color=ink_on(c), fontsize=10)
    c = blue_for(r["prevalence"], p_min, p_max)
    ax.add_patch(plt.Rectangle((1.0, y - 0.42), 0.92, 0.84, facecolor=c, edgecolor=SURFACE, lw=2))
    ax.text(1.46, y, f"{r['prevalence']*100:.1f}%", ha="center", va="center", color=ink_on(c), fontsize=10)
    cls = r["drug_class"]
    c = CLASS_COLOR[cls]
    ax.add_patch(plt.Rectangle((2.0, y - 0.42), 0.92, 0.84, facecolor=c, edgecolor=SURFACE, lw=2))
    ax.text(2.46, y, f"{r['gene']}", ha="center", va="center", color=ink_on(c), fontsize=10, fontweight="bold")
    label = r["drug"] + ("*" if r["is_sparse"] else "")
    ax.text(-0.12, y, label, ha="right", va="center", color=INK, fontsize=10, fontweight="bold")

for label, x in [("sample size n", 0.46), ("prevalence", 1.46), ("protein/region", 2.46)]:
    ax.text(x, n - 0.35, label, ha="center", va="bottom", color=INK2, fontsize=10)

for i, cls in enumerate(CLASS_ORDER):
    x0 = 0.0 + i * 0.6
    ax.add_patch(plt.Rectangle((x0, -1.5), 0.16, 0.5, facecolor=CLASS_COLOR[cls], edgecolor=SURFACE, lw=1.5))
    ax.text(x0 + 0.2, -1.25, cls, ha="left", va="center", color=INK, fontsize=9)

ax.set_xlim(-1.4, 3.0)
ax.set_ylim(-1.8, n + 0.2)
ax.axis("off")
ax.set_title("Fig 1 - Data-attribute matrix for 24 drugs (sample size / prevalence / protein region)",
             loc="left", color=INK, fontsize=12, pad=12)
fig.text(0.01, 0.005, "Deeper blue = larger value; the protein-region column is colored by drug class. * = sparse task (CAB, LEN).", color=INK3, fontsize=8.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "drug_task_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved", (FIG_DIR / "drug_task_matrix.png").relative_to(PROJECT_ROOT))

## 6 · Figure 2: data-flow Sankey

This figure shows how HIVDB-derived sequences are organized by gene region, drug class, individual drug tasks, and final task instances. It emphasizes that one sequence can contribute to multiple drug-specific prediction tasks, which is central to the long-format task definition. The four gene regions (PR, RT, IN, CA) feed the five drug classes.

In [ ]:
def sankey_band(ax, x0, x1, y0a, y0b, y1a, y1b, color, alpha=0.55):
    verts = [
        (x0, y0a), (x0 + (x1 - x0) * 0.5, y0a), (x0 + (x1 - x0) * 0.5, y1a), (x1, y1a),
        (x1, y1b), (x0 + (x1 - x0) * 0.5, y1b), (x0 + (x1 - x0) * 0.5, y0b), (x0, y0b), (x0, y0a),
    ]
    codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
             MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CLOSEPOLY]
    ax.add_patch(PathPatch(MplPath(verts, codes), facecolor=color, edgecolor="none", alpha=alpha))

fig, ax = plt.subplots(figsize=(10.5, 7.4))
GAP = 0.5
total = per_drug["n_valid"].sum()
scale = 10.0 / total
x = {"src": 0.0, "gene": 1.0, "cls": 2.0, "drug": 3.0}
w = 0.16
GENE_ORDER = ["PR", "RT", "IN", "CA"]

ax.add_patch(plt.Rectangle((x["src"], 0), w, 10.0, facecolor=INK2, edgecolor=SURFACE, lw=1))
ax.text(x["src"] - 0.05, 5.0, f"HIVDB\ntask instances\n{int(total)}", ha="right", va="center", color=INK, fontsize=9.5)

gene_tot = per_drug.groupby("gene", observed=True)["n_valid"].sum().reindex(GENE_ORDER).dropna()
gene_y, cur = {}, 0.0
for g, v in gene_tot.items():
    h = v * scale
    gene_y[g] = (cur, cur + h)
    ax.add_patch(plt.Rectangle((x["gene"], cur), w, h, facecolor=INK3, edgecolor=SURFACE, lw=1))
    ax.text(x["gene"] + w + 0.04, cur + h / 2, f"{g}\n{int(v)}", ha="left", va="center", color=INK, fontsize=9)
    cur += h + GAP

src_cursor = 0.0
for g, v in gene_tot.items():
    h = v * scale
    sankey_band(ax, x["src"] + w, x["gene"], src_cursor, src_cursor + h,
                gene_y[g][0], gene_y[g][1], INK3, alpha=0.35)
    src_cursor += h

cls_tot = per_drug.groupby("drug_class", observed=True)["n_valid"].sum().reindex(CLASS_ORDER)
cls_y, cur = {}, 0.0
for cls in CLASS_ORDER:
    v = cls_tot[cls]
    h = v * scale
    cls_y[cls] = (cur, cur + h)
    ax.add_patch(plt.Rectangle((x["cls"], cur), w, h, facecolor=CLASS_COLOR[cls], edgecolor=SURFACE, lw=1))
    ax.text(x["cls"] + w + 0.04, cur + h / 2, f"{cls}\n{int(v)}", ha="left", va="center", color=INK, fontsize=9)
    cur += h + GAP

gene_cursor = {g: gene_y[g][0] for g in gene_y}
for cls in CLASS_ORDER:
    g = CLASS_GENE[cls]
    v = cls_tot[cls]
    h = v * scale
    sankey_band(ax, x["gene"] + w, x["cls"], gene_cursor[g], gene_cursor[g] + h,
                cls_y[cls][0], cls_y[cls][1], CLASS_COLOR[cls], alpha=0.35)
    gene_cursor[g] += h

cur = 0.0
cls_cursor = {c: cls_y[c][0] for c in cls_y}
for cls in CLASS_ORDER:
    sub = per_drug[per_drug["drug_class"] == cls].sort_values("n_valid", ascending=False)
    for _, r in sub.iterrows():
        h = r["n_valid"] * scale
        ax.add_patch(plt.Rectangle((x["drug"], cur), w, h, facecolor=CLASS_COLOR[cls], edgecolor=SURFACE, lw=1))
        label = r["drug"] + ("*" if r["is_sparse"] else "")
        ax.text(x["drug"] + w + 0.04, cur + h / 2, label, ha="left", va="center", color=INK, fontsize=8.5)
        sankey_band(ax, x["cls"] + w, x["drug"], cls_cursor[cls], cls_cursor[cls] + h,
                    cur, cur + h, CLASS_COLOR[cls], alpha=0.32)
        cls_cursor[cls] += h
        cur += h + GAP * 0.35

ax.set_xlim(-1.0, 3.9)
ax.set_ylim(-0.4, max(cur, 10.5) + 0.2)
ax.axis("off")
ax.set_title("Fig 2 - Data flow from HIVDB sequences to 24 drug tasks", loc="left", color=INK, fontsize=12, pad=10)
fig.text(0.01, 0.005, "Band width proportional to valid task instances; color = drug class (PR->PI, RT->NRTI/NNRTI, IN->INI, CA->CAI). * = sparse task.", color=INK3, fontsize=8.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "data_flow_sankey.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved", (FIG_DIR / "data_flow_sankey.png").relative_to(PROJECT_ROOT))

## 7 · Figure 3: sequence-QC card panel

This figure presents class-level sequence quality-control results as compact cards. Each card reports the number of unique sequences, length range, non-standard amino-acid frequency, and duplicate sequence count. Green markers indicate checks that pass directly, while yellow markers indicate cases handled during feature construction.

In [ ]:
GOOD = "#0ca30c"
WARN = "#fab219"

ncards = len(seq_qc)
ncols = 3
nrows = int(np.ceil(ncards / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(11, 3.3 * nrows))
axes = np.atleast_1d(axes).ravel()
for ax, (_, r) in zip(axes, seq_qc.iterrows()):
    cls = r["drug_class"]
    ax.axis("off")
    ax.add_patch(plt.Rectangle((0.02, 0.02), 0.96, 0.96, transform=ax.transAxes,
                               facecolor=SURFACE, edgecolor=CLASS_COLOR[cls], lw=2.5))
    ax.text(0.08, 0.86, f"{cls}", transform=ax.transAxes, color=CLASS_COLOR[cls], fontsize=15, fontweight="bold")
    ax.text(0.92, 0.87, f"gene {r['gene']}", transform=ax.transAxes, color=INK2, fontsize=9.5, ha="right")

    len_ok = r["n_len_outliers"] == 0 or (r["len_max"] - r["len_min"] <= 1)
    lines = [
        (f"unique sequences  {int(r['n_isolates'])}", INK, None),
        (f"length range  {int(r['len_min'])}-{int(r['len_max'])}  (mode {int(r['len_mode'])})",
         INK2, GOOD if len_ok else WARN),
        (f"seqs with non-std AA  {int(r['n_seq_with_nonstd_aa'])} ({r['pct_seq_with_nonstd_aa']}%)",
         INK2, GOOD if r["n_seq_with_nonstd_aa"] == 0 else WARN),
        (f"duplicate sequences  {int(r['n_duplicate_sequences'])}", INK2, None),
    ]
    y = 0.66
    for text, color, dot in lines:
        if dot is not None:
            ax.add_patch(plt.Circle((0.11, y + 0.012), 0.018, transform=ax.transAxes, facecolor=dot, edgecolor="none"))
            tx = 0.16
        else:
            tx = 0.08
        ax.text(tx, y, text, transform=ax.transAxes, color=color, fontsize=10.5, va="center")
        y -= 0.17

for ax in axes[ncards:]:
    ax.axis("off")

fig.suptitle("Fig 3 - Sequence-QC cards for the five drug classes", x=0.5, y=1.0, color=INK, fontsize=12, ha="center")
fig.text(0.01, -0.02, "Green dot = pass; yellow dot = handle during feature construction (1-position length diff or non-standard AA).",
         color=INK3, fontsize=8.5)
fig.tight_layout()
fig.savefig(FIG_DIR / "sequence_qc_panel.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved", (FIG_DIR / "sequence_qc_panel.png").relative_to(PROJECT_ROOT))

## 8 · Sparse-task diagnostics: CAB and LEN

INI and CAI are now first-class members of the unified `task_table.csv`. This module provides focused diagnostics for the two sparse tasks flagged above — CAB (integrase, minority class = 12) and LEN (capsid, n = 140 with ~70% duplicate sequences). It records how the raw Stanford HIVDB integrase- and capsid-inhibitor datasets were sourced and reports a duplicate-sequence-aware baseline so downstream notebooks know why these two tasks carry the sparse flag.

The remaining INI tasks (RAL, EVG, DTG, BIC) have sufficient sample size and are treated like any other primary-benchmark drug. The diagnostics below quantify the gap between optimistic random-split performance and stricter duplicate-aware evaluation, which is exactly why CAB and LEN are flagged rather than dropped.

In [12]:
from urllib.request import Request, urlopen

HIVDB_RAW_DIR = PROJECT_ROOT / "work" / "hivdb_raw"
EXTENDED_DIR = RESULTS_DIR / "ini_cai_mini_pipeline"
HIVDB_RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTENDED_DIR.mkdir(parents=True, exist_ok=True)

EXTENDED_URLS = {
    "INI": "https://hivdb.stanford.edu/download/GenoPhenoDatasets/INI_DataSet.txt",
    "CAI": "https://hivdb.stanford.edu/download/GenoPhenoDatasets/CAI_DataSet.txt",
}

def project_path(path: Path) -> str:
    return "/" + path.relative_to(PROJECT_ROOT).as_posix()

for cls, url in EXTENDED_URLS.items():
    dest = HIVDB_RAW_DIR / f"{cls}_DataSet.txt"
    if not dest.exists():
        req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(req, timeout=120) as response:
            dest.write_bytes(response.read())
    print(f"{cls}: {project_path(dest)}")

INI: /work/hivdb_raw/INI_DataSet.txt
CAI: /work/hivdb_raw/CAI_DataSet.txt


### 8.1 · Extended category data availability

The mini pipeline writes compact summary artifacts under `results/notebooks/01_data/ini_cai_mini_pipeline/`. The table below reports how many usable binary prediction instances are available after parsing the INI and CAI HIVDB files.

### 8.2 · Random-split and duplicate-sequence-aware baseline check

The extended mini pipeline uses a lightweight position-wise amino-acid one-hot representation with class-balanced logistic regression. This check is not a replacement for the full downstream modeling pipeline. It is used here only to determine whether INI and CAI behave like feasible extension tasks or sparse stress-test cases.

In [13]:
metrics_path = EXTENDED_DIR / "random_vs_group_baseline_metrics.csv"
if metrics_path.exists():
    extended_metrics = pd.read_csv(metrics_path)
    display_cols = [
        "drug_class", "drug", "n", "prevalence", "model_runnable",
        "random_auroc_mean", "group_auroc_mean", "auroc_group_minus_random",
        "random_balanced_accuracy_mean", "group_balanced_accuracy_mean", "bacc_group_minus_random",
        "skip_reason",
    ]
    display_cols = [c for c in display_cols if c in extended_metrics.columns]
    extended_metrics[display_cols]
else:
    pd.DataFrame({"message": ["Run /test_ini_cai_mini_pipeline.py to generate baseline metrics."]})

### 8.3 · Interpretation for the main benchmark

The INI tasks RAL, EVG, DTG, and BIC are feasible extension tasks. CAB is too small for stable model evaluation, and LEN has a small sample size with high duplicate-sequence structure. These two tasks are useful as sparse stress-test cases but should not be averaged into the primary benchmark.

The extended categories therefore support the existing methodological framing: random-split performance can look strong, while sparse or redundancy-heavy tasks require stricter evaluation. INI and CAI are retained as an extended data layer rather than merged into the primary PI/NRTI/NNRTI task table.

In [14]:
task_dist_path = EXTENDED_DIR / "task_distribution.csv"
if task_dist_path.exists():
    extended_task_distribution = pd.read_csv(task_dist_path)
else:
    extended_task_distribution = pd.DataFrame({
        "message": ["Run /test_ini_cai_mini_pipeline.py to generate this extended summary."]
    })

extended_task_distribution

,drug_class,drug,n,n_resistant,n_unique_sequences,phenotype_min,phenotype_median,phenotype_max,n_susceptible,prevalence,duplicate_sequence_fraction
0,CAI,LEN,140,111,42,0.5,16.50,100.0,29,0.792857,0.700000
1,INI,BIC,287,51,195,0.1,1.30,66.0,236,0.177700,0.320557
2,INI,CAB,64,52,14,0.6,8.60,100.0,12,0.812500,0.781250
3,INI,DTG,370,72,278,0.1,1.10,100.0,298,0.194595,0.248649
4,INI,EVG,754,377,610,0.6,2.95,100.0,377,0.500000,0.190981
5,INI,RAL,753,311,637,0.3,1.30,100.0,442,0.413015,0.154050


## 9 · Summary

This notebook creates the standardized data foundation for the primary benchmark and records INI/CAI as an extended data scope.

- The primary unified task table covers **3 drug classes and 18 drugs**.
- PR sequences have length 99, while RT sequences are approximately 240 or 318 amino acids depending on drug class context.
- Resistance prevalence varies widely across drugs, with low-prevalence or sparse tasks such as DDI, TDF, and RPV requiring careful downstream interpretation.
- Sequence lengths are largely uniform, and non-standard amino acids occur in a minority of sequences, allowing mutation encoding to align most primary-benchmark sequences to reference positions directly.
- INI and CAI are downloaded and summarized as extended categories. INI contains several feasible follow-up tasks, while CAB and LEN are better treated as sparse stress-test cases.
- The saved artifacts in `results/notebooks/01_data/` provide the shared inputs for feature construction, model comparison, OOD evaluation, interpretability analyses, and extended-category inspection.

The next notebook, `02_indistribution_benchmark.ipynb`, evaluates primary-benchmark model performance under standard in-distribution settings.